## Contenido

1. Importar librerías
2. Cargar datos procesados
3. Mostrar métricas
4. Mostrar gráficos
5. Conclusiones

## 1. Importar librerías

In [117]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import folium
import seaborn as sns

print('Librerias cargadas correctamente')
print(f'Version de pandas: {pd.__version__}')
print(f'Version de numpy: {np.__version__}')
print(f'Version de folium: {folium.__version__}')

Librerias cargadas correctamente
Version de pandas: 2.2.2
Version de numpy: 2.0.2
Version de folium: 0.20.0


In [118]:
from google.colab import files

In [119]:
## Exploracion de datos: Convertir unidades de UTM al sistema WGS84 (EPSG:4326)
!pip install pyproj

## 2. Carga de datos procesados

Datos subidos por separado con el ID del archivo csv.

In [120]:
# gdown descarga archivos de Google Drive a partir de su ID, sin necesidad de login
#!pip install -q gdown

import gdown, os

FILE_ID = '188DSux_eIeeuVP5hUI0A5D09OedPrHIb'  # CSV de IPASH-Suelo
CSV_PATH_SUELO = 'IPASH_Componente Ambiental_Suelo.csv'

# Evita volver a descargar si el archivo ya existe en el entorno de Colab
if not os.path.exists(CSV_PATH_SUELO):
    gdown.download(id=FILE_ID, output=CSV_PATH_SUELO, quiet=False)

print('\nCSV listo en:', CSV_PATH_SUELO)


CSV listo en: IPASH_Componente Ambiental_Suelo.csv


In [121]:
# gdown descarga archivos de Google Drive a partir de su ID, sin necesidad de login
#!pip install -q gdown

import gdown, os

FILE_ID = '1ti1hH1mgqYEYa_CQULU-rtPem1hCQ7yf'  # CSV de IPASH-Sedimento
CSV_PATH_SEDIMENTO = 'IPASH_Componente Ambiental_Sedimento.csv'

# Evita volver a descargar si el archivo ya existe en el entorno de Colab
if not os.path.exists(CSV_PATH_SEDIMENTO):
    gdown.download(id=FILE_ID, output=CSV_PATH_SEDIMENTO, quiet=False)

print('\nCSV listo en:', CSV_PATH_SEDIMENTO)


CSV listo en: IPASH_Componente Ambiental_Sedimento.csv


In [122]:
# gdown descarga archivos de Google Drive a partir de su ID, sin necesidad de login
#!pip install -q gdown

import gdown, os

FILE_ID = '1Z0rZlIq2pB_SGWsIx-HfTO4IEs6FZdaZ'  # CSV de IPASH-Agua
CSV_PATH_AGUA = 'IPASH_Componente Ambiental_Agua.csv'

# Evita volver a descargar si el archivo ya existe en el entorno de Colab
if not os.path.exists(CSV_PATH_AGUA):
    gdown.download(id=FILE_ID, output=CSV_PATH_AGUA, quiet=False)

print('CSV listo en:', CSV_PATH_AGUA)

CSV listo en: IPASH_Componente Ambiental_Agua.csv


## 3. Procesamiento de datos (limpieza)

### 3.1 Datos de SUELO

Orden:
1. Corregir ubicación --> Utilizar coordenadas
2. Corregir Fecha
3. Corregir Hora (formato 24h o mantener en AM y PM)
4. Crear columna de Valor_num
5. Extraer solo columna que se van a utilizar: Etapa, Componente ambiental,
Nombre del punto, Coordenadas, Fecha, Hora, Valor_num, Parámetro, Unidad de medida

In [123]:
# Nombre para CSV_PATH_SUELO

df_suelo = pd.read_csv(CSV_PATH_SUELO)

df_suelo.head()

,Número de informe,Nombre de la Evaluación,Etapa,Componente ambiental,Procedencia de la Muestra,Procedencia Especifica de la Muestra,Nombre del punto,Este,Norte,Altitud,...,Descripción de ubicación,Tipo de muestra,Tipo de análisis,Etapa de extracción secuencial,Caracterización de la muestra,Fecha,Hora,Valor,Parámetro,Unidad de medida
0,INFORME N° 00003-2017-OEFA/DEAM-SSIM,Identificación de Pasivo Ambiental del Subsect...,Primer monitoreo,Suelo,Suelo,Área de potencial interés,F05447-SU01,478216,9526380,275,...,Muestra puntual de suelo tomada a 1.3 m en dir...,Simple,No aplica,No aplica,Ensayo por cromatografía - HTP,11/7/17,11:12:00 AM,514,Fracciones de Hidrocarburos F2 (C10-C28),mg/Kg
1,INFORME N° 00003-2017-OEFA/DEAM-SSIM,Identificación de Pasivo Ambiental del Subsect...,Primer monitoreo,Suelo,Suelo,Área de potencial interés,F05447-SU01,478216,9526380,275,...,Muestra puntual de suelo tomada a 1.3 m en dir...,Simple,No aplica,No aplica,Ensayo por cromatografía - HTP,11/7/17,11:12:00 AM,3795,Fracciones de Hidrocarburos F3 (C28-C40),mg/Kg
2,INFORME N° 00003-2017-OEFA/DEAM-SSIM,Identificación de Pasivo Ambiental del Subsect...,Primer monitoreo,Suelo,Suelo,Área de potencial interés,F05447-SU01,478216,9526380,275,...,Muestra puntual de suelo tomada a 1.3 m en dir...,Simple,No aplica,No aplica,Ensayo por cromatografía - HTP,11/7/17,11:12:00 AM,<0.3,Fracción de Hidrocarburos F1 (C5-C10),mg/Kg
3,INFORME N° 00003-2017-OEFA/DEAM-SSIM,Identificación de Pasivo Ambiental del Subsect...,Primer monitoreo,Suelo,Suelo,Área de potencial interés,F05447-SU01,478216,9526380,275,...,Muestra puntual de suelo tomada a 1.3 m en dir...,Simple,No aplica,No aplica,Ensayo por cromatografía - HTP,11/7/17,11:12:00 AM,4310,Hidrocarburos totales de petróleo (C5-C40),mg/Kg
4,INFORME N° 00003-2017-OEFA/DEAM-SSIM,Identificación de Pasivo Ambiental del Subsect...,Primer monitoreo,Suelo,Suelo,Área de potencial interés,F05447-SU02,478224,9526389,275,...,Muestra puntual de suelo tomada a 8.4 m en dir...,Simple,No aplica,No aplica,Ensayo por cromatografía - HTP,11/7/17,11:21:00 AM,<5.00,Fracciones de Hidrocarburos F2 (C10-C28),mg/Kg


In [124]:
## Limpieza de datos: Tratamiento de valores faltantes

#Con esta linea de códigos detectamos columnas con datos faltantes
df_suelo_nulos = df_suelo.isnull().sum()
print('Valores nulos por columna: Matriz Suelo')
print(df_suelo_nulos[df_suelo_nulos > 0] if df_suelo_nulos.any() else 'No hay valores nulos')

Valores nulos por columna: Matriz Suelo
Tipo de análisis                  357
Etapa de extracción secuencial    357
dtype: int64


In [125]:
##Exploración de datos: mapa de puntos de monitoreo por matriz
# Conversión de Unidades

from pyproj import Transformer

def utm_to_latlon_dynamic(row):
    # Determinar el sistema de referencias de coordenadas (CRS) UTM a partir de la columna 'Zona'
    # Suponiento el Hemisferio Sur para Perú (EPSG:327XX)
    utm_crs = f"epsg:327{int(row['Zona'])}" #Como las muestras pueden venir de diferentes UTM, esta función genera el EPSG dinámico
    # Transformer.from_crs expects source_crs, target_crs
    # always_xy=True ensures output is (longitude, latitude)
    transformer = Transformer.from_crs(utm_crs, "epsg:4326", always_xy=True)

    # 'Este' is Easting, 'Norte' is Northing
    lon_wgs84, lat_wgs84 = transformer.transform(row['Este'], row['Norte'])
    return pd.Series([lat_wgs84, lon_wgs84])

In [126]:
# Estas lineas son para aplicar la conversión dinámica al DataFrame.
# Despues hay que asegurarse que las columnas de 'Zona', 'Este', 'Norte' sean numéricas y esta linea gestiona posibles errores
for current_df in [df_suelo]:
    cols_to_process = []
    if 'Este' in current_df.columns:
        cols_to_process.append('Este')
    if 'Norte' in current_df.columns:
        cols_to_process.append('Norte')
    if 'Zona' in current_df.columns: # Solo procesar 'Zona' si existe
        cols_to_process.append('Zona')

    for col in cols_to_process:
        current_df[col] = pd.to_numeric(current_df[col], errors='coerce')

    if cols_to_process: # Solo aplicar dropna si hay columnas a considerar
        current_df.dropna(subset=cols_to_process, inplace=True)

# Aplicar la conversión de UTM a Lat/Lon
# Solo aplicar si 'Zona', 'Este' y 'Norte' existen en el DataFrame
if all(col in df_suelo.columns for col in ['Este', 'Norte', 'Zona']):
    df_suelo[['lat', 'lon']] = df_suelo.apply(utm_to_latlon_dynamic, axis=1) #Esta linea crea las coordenadas en el formato que son necesarias para Folium (suelo)
else:
    print("Las columnas 'Este', 'Norte' o 'Zona' no están disponibles en df_suelo, no se aplicó la conversión UTM a Lat/Lon.")

print("Coordenadas convertidas a WGS84 para suelo (si las columnas estaban disponibles).")

# Mostrar el head de 'suelo' con las nuevas coordenadas para verificación
if 'lat' in df_suelo.columns:
    display(df_suelo[['Nombre del punto', 'Este', 'Norte', 'Zona', 'lat', 'lon']].head())
else:
    print("No se pudieron generar las columnas 'lat' y 'lon' para df_suelo.")

Coordenadas convertidas a WGS84 para suelo (si las columnas estaban disponibles).


,Nombre del punto,Este,Norte,Zona,lat,lon
0,F05447-SU01,478216,9526380,17,-4.284883,-81.196312
1,F05447-SU01,478216,9526380,17,-4.284883,-81.196312
2,F05447-SU01,478216,9526380,17,-4.284883,-81.196312
3,F05447-SU01,478216,9526380,17,-4.284883,-81.196312
4,F05447-SU02,478224,9526389,17,-4.284802,-81.196240


In [127]:
# Crear columna Valor_num --> Correción de los valores numéricos

#Segundo: limpiamos los valores y lo reemplazamos por LD/2
import re
import numpy as np
import pandas as pd # Added for pd.read_csv and pd.Int64Dtype()

def limpiar_valor(v):
  '''
  Convierte un valor analítico a numero_
  - Si contiene '<', extra el LD y lo divide entre 2 (convención USEPA)
  - Reemplaza coma decimal por punto
  - Si no puede convertirse, devuelve NaN
  '''

  v = str(v).strip()
  v = v.replace(',','.')     #linea para cambiar la coma decimal por el punto
  if v.startswith('<'):
    try:
      lod = float(v.replace('<', '').strip())   #linea para extraer el número despues del '<'
      return lod/2
    except ValueError:
      return np.nan
  try:
    return float(v)
  except ValueError:
    return np.nan

#Tercero: aplicar la función a toda la columna Valor de cada dataset
df_suelo['Valor_num'] = df_suelo['Valor'].apply(limpiar_valor)

#Cuarto: imprimir resultado
print('Columna Valor_num creada')
print(f'Valores numéricos válidos (suelo): {df_suelo['Valor_num'].notna().sum()}')
print(f'Valores no convertibles (NaN) (suelo): {df_suelo['Valor_num'].isna().sum()}')

Columna Valor_num creada
Valores numéricos válidos (suelo): 999
Valores no convertibles (NaN) (suelo): 0


In [128]:
# FECHA viene como MM/DD/YY (int) -> lo convertimos a datetime real
df_suelo['Fecha'] = pd.to_datetime(df_suelo['Fecha'], format='%m/%d/%y', errors='coerce')

# Hora viene en formato AM/PM --> convertir a formato 0 a 23h
df_suelo['Hora'] = pd.to_datetime(df_suelo['Hora'], format='mixed', errors='coerce').dt.strftime('%H:%M')

In [129]:
df_suelo.head()

,Número de informe,Nombre de la Evaluación,Etapa,Componente ambiental,Procedencia de la Muestra,Procedencia Especifica de la Muestra,Nombre del punto,Este,Norte,Altitud,...,Etapa de extracción secuencial,Caracterización de la muestra,Fecha,Hora,Valor,Parámetro,Unidad de medida,lat,lon,Valor_num
0,INFORME N° 00003-2017-OEFA/DEAM-SSIM,Identificación de Pasivo Ambiental del Subsect...,Primer monitoreo,Suelo,Suelo,Área de potencial interés,F05447-SU01,478216,9526380,275,...,No aplica,Ensayo por cromatografía - HTP,2017-11-07,11:12,514,Fracciones de Hidrocarburos F2 (C10-C28),mg/Kg,-4.284883,-81.196312,514.00
1,INFORME N° 00003-2017-OEFA/DEAM-SSIM,Identificación de Pasivo Ambiental del Subsect...,Primer monitoreo,Suelo,Suelo,Área de potencial interés,F05447-SU01,478216,9526380,275,...,No aplica,Ensayo por cromatografía - HTP,2017-11-07,11:12,3795,Fracciones de Hidrocarburos F3 (C28-C40),mg/Kg,-4.284883,-81.196312,3795.00
2,INFORME N° 00003-2017-OEFA/DEAM-SSIM,Identificación de Pasivo Ambiental del Subsect...,Primer monitoreo,Suelo,Suelo,Área de potencial interés,F05447-SU01,478216,9526380,275,...,No aplica,Ensayo por cromatografía - HTP,2017-11-07,11:12,<0.3,Fracción de Hidrocarburos F1 (C5-C10),mg/Kg,-4.284883,-81.196312,0.15
3,INFORME N° 00003-2017-OEFA/DEAM-SSIM,Identificación de Pasivo Ambiental del Subsect...,Primer monitoreo,Suelo,Suelo,Área de potencial interés,F05447-SU01,478216,9526380,275,...,No aplica,Ensayo por cromatografía - HTP,2017-11-07,11:12,4310,Hidrocarburos totales de petróleo (C5-C40),mg/Kg,-4.284883,-81.196312,4310.00
4,INFORME N° 00003-2017-OEFA/DEAM-SSIM,Identificación de Pasivo Ambiental del Subsect...,Primer monitoreo,Suelo,Suelo,Área de potencial interés,F05447-SU02,478224,9526389,275,...,No aplica,Ensayo por cromatografía - HTP,2017-11-07,11:21,<5.00,Fracciones de Hidrocarburos F2 (C10-C28),mg/Kg,-4.284802,-81.196240,2.50


In [130]:
# Columnas finales que se van a usar para SUELO
# SI FALTAN COLUMNAS AÑADIR AQUÍ

final_cols_suelo = ['Etapa','Componente ambiental','Nombre del punto','lat','lon','Fecha','Hora','Valor_num','Parámetro','Unidad de medida']

# Asegurarse de que todas las columnas finales existen antes de seleccionar
missing_cols = [col for col in final_cols_suelo if col not in df_suelo.columns]
if missing_cols:
    print(f"Advertencia: Las siguientes columnas faltan en df_suelo y no pueden ser seleccionadas: {missing_cols}")

# Seleccionar solo las columnas deseadas que existen
df_suelo = df_suelo[[col for col in final_cols_suelo if col in df_suelo.columns]]

df_suelo.head()

,Etapa,Componente ambiental,Nombre del punto,lat,lon,Fecha,Hora,Valor_num,Parámetro,Unidad de medida
0,Primer monitoreo,Suelo,F05447-SU01,-4.284883,-81.196312,2017-11-07,11:12,514.00,Fracciones de Hidrocarburos F2 (C10-C28),mg/Kg
1,Primer monitoreo,Suelo,F05447-SU01,-4.284883,-81.196312,2017-11-07,11:12,3795.00,Fracciones de Hidrocarburos F3 (C28-C40),mg/Kg
2,Primer monitoreo,Suelo,F05447-SU01,-4.284883,-81.196312,2017-11-07,11:12,0.15,Fracción de Hidrocarburos F1 (C5-C10),mg/Kg
3,Primer monitoreo,Suelo,F05447-SU01,-4.284883,-81.196312,2017-11-07,11:12,4310.00,Hidrocarburos totales de petróleo (C5-C40),mg/Kg
4,Primer monitoreo,Suelo,F05447-SU02,-4.284802,-81.196240,2017-11-07,11:21,2.50,Fracciones de Hidrocarburos F2 (C10-C28),mg/Kg


### 3.2 Datos de SEDIMENTO

Orden:
1. Corregir ubicación --> Utilizar coordenadas
2. Corregir Fecha
3. Corregir Hora (formato 24h o mantener en AM y PM)
4. Crear columna de Valor_num
5. Extraer solo columna que se van a utilizar: Etapa, Componente ambiental,
Nombre del punto, Coordenadas, Fecha, Hora, Valor_num, Parámetro, Unidad de medida

In [131]:
# Nombre para CSV_PATH_SEDIMENTO

df_sedimento = pd.read_csv(CSV_PATH_SEDIMENTO)

df_sedimento.head()

,Número de informe,Nombre de la Evaluación,Etapa,Componente ambiental,Procedencia de la Muestra,Procedencia Especifica de la Muestra,Nombre del punto,Este,Norte,Altitud,...,Datum,Descripción de ubicación,Tipo de muestra,Tipo de análisis,Etapa de extracción secuencial,Fecha,Hora,Valor,Parámetro,Unidad de medida
0,INFORME N° 00580-2019-OEFA/DEAM-SSIM,Identificación de Pasivo Ambiental del Subsect...,Primer monitoreo,Sedimento,Sedimento continental,Área de potencial interés,F05898_SD01,468461,9507737,0,...,WGS 84,Muestra puntual de sedimentos tomada a 4m en d...,Simple,Ensayo por cromatografía - HTP,No aplica,10/6/19,10:07:00,<1.9,Fracciones de Hidrocarburos F1 (C6-C10),mg/Kg
1,INFORME N° 00580-2019-OEFA/DEAM-SSIM,Identificación de Pasivo Ambiental del Subsect...,Primer monitoreo,Sedimento,Sedimento continental,Área de potencial interés,F05898_SD01,468461,9507737,0,...,WGS 84,Muestra puntual de sedimentos tomada a 4m en d...,Simple,Ensayo por cromatografía - HTP,No aplica,10/6/19,10:07:00,<6.8,Fracciones de Hidrocarburos F2 (C10-C28),mg/Kg
2,INFORME N° 00580-2019-OEFA/DEAM-SSIM,Identificación de Pasivo Ambiental del Subsect...,Primer monitoreo,Sedimento,Sedimento continental,Área de potencial interés,F05898_SD01,468461,9507737,0,...,WGS 84,Muestra puntual de sedimentos tomada a 4m en d...,Simple,Ensayo por cromatografía - HTP,No aplica,10/6/19,10:07:00,<6.8,Fracciones de Hidrocarburos F3 (C28-C40),mg/Kg
3,INFORME N° 00580-2019-OEFA/DEAM-SSIM,Identificación de Pasivo Ambiental del Subsect...,Primer monitoreo,Sedimento,Sedimento continental,Área de potencial interés,F05898_SD01,468461,9507737,0,...,WGS 84,Muestra puntual de sedimentos tomada a 4m en d...,Simple,Ensayo por cromatografía - PAHS,No aplica,10/6/19,10:07:00,<0.0054,Acenafteno,mg/Kg
4,INFORME N° 00580-2019-OEFA/DEAM-SSIM,Identificación de Pasivo Ambiental del Subsect...,Primer monitoreo,Sedimento,Sedimento continental,Área de potencial interés,F05898_SD01,468461,9507737,0,...,WGS 84,Muestra puntual de sedimentos tomada a 4m en d...,Simple,Ensayo por cromatografía - PAHS,No aplica,10/6/19,10:07:00,<0.0054,Acenaftileno,mg/Kg


In [132]:
## Limpieza de datos: Tratamiento de valores faltantes

#Con esta linea de códigos detectamos columnas con datos faltantes
df_sedimento_nulos = df_sedimento.isnull().sum()
print('Valores nulos por columna: Matriz Sedimento')
print(df_sedimento_nulos[df_sedimento_nulos > 0] if df_sedimento_nulos.any() else 'No hay valores nulos')

Valores nulos por columna: Matriz Sedimento
No hay valores nulos


In [133]:
##Exploración de datos: mapa de puntos de monitoreo por matriz
# Conversión de Unidades

from pyproj import Transformer

def utm_to_latlon_dynamic(row):
    # Determinar el sistema de referencias de coordenadas (CRS) UTM a partir de la columna 'Zona'
    # Suponiento el Hemisferio Sur para Perú (EPSG:327XX)
    utm_crs = f"epsg:327{int(row['Zona'])}" #Como las muestras pueden venir de diferentes UTM, esta función genera el EPSG dinámico
    # Transformer.from_crs expects source_crs, target_crs
    # always_xy=True ensures output is (longitude, latitude)
    transformer = Transformer.from_crs(utm_crs, "epsg:4326", always_xy=True)

    # 'Este' is Easting, 'Norte' is Northing
    lon_wgs84, lat_wgs84 = transformer.transform(row['Este'], row['Norte'])
    return pd.Series([lat_wgs84, lon_wgs84])

In [134]:
# Estas lineas son para aplicar la conversión dinámica al DataFrame.
# Despues hay que asegurarse que las columnas de 'Zona', 'Este', 'Norte' sean numéricas y esta linea gestiona posibles errores
for current_df in [df_sedimento]:
    cols_to_process = []
    if 'Este' in current_df.columns:
        cols_to_process.append('Este')
    if 'Norte' in current_df.columns:
        cols_to_process.append('Norte')
    if 'Zona' in current_df.columns: # Solo procesar 'Zona' si existe
        cols_to_process.append('Zona')

    for col in cols_to_process:
        current_df[col] = pd.to_numeric(current_df[col], errors='coerce')

    if cols_to_process: # Solo aplicar dropna si hay columnas a considerar
        current_df.dropna(subset=cols_to_process, inplace=True)

# Aplicar la conversión de UTM a Lat/Lon
# Solo aplicar si 'Zona', 'Este' y 'Norte' existen en el DataFrame
if all(col in df_sedimento.columns for col in ['Este', 'Norte', 'Zona']):
    df_sedimento[['lat', 'lon']] = df_sedimento.apply(utm_to_latlon_dynamic, axis=1) #Esta linea crea las coordenadas en el formato que son necesarias para Folium (sedimento)
else:
    print("Las columnas 'Este', 'Norte' o 'Zona' no están disponibles en df_sedimento, no se aplicó la conversión UTM a Lat/Lon.")

print("Coordenadas convertidas a WGS84 para sedimento (si las columnas estaban disponibles).")

# Mostrar el head de 'sedimento' con las nuevas coordenadas para verificación
if 'lat' in df_sedimento.columns:
    display(df_sedimento[['Nombre del punto', 'Este', 'Norte', 'Zona', 'lat', 'lon']].head())
else:
    print("No se pudieron generar las columnas 'lat' y 'lon' para df_sedimento.")

Coordenadas convertidas a WGS84 para sedimento (si las columnas estaban disponibles).


,Nombre del punto,Este,Norte,Zona,lat,lon
0,F05898_SD01,468461,9507737,17,-4.453513,-81.284285
1,F05898_SD01,468461,9507737,17,-4.453513,-81.284285
2,F05898_SD01,468461,9507737,17,-4.453513,-81.284285
3,F05898_SD01,468461,9507737,17,-4.453513,-81.284285
4,F05898_SD01,468461,9507737,17,-4.453513,-81.284285


In [135]:
# Crear columna Valor_num --> Correción de los valores numéricos

#Segundo: limpiamos los valores y lo reemplazamos por LD/2
import re
import numpy as np
import pandas as pd # Added for pd.read_csv and pd.Int64Dtype()

def limpiar_valor(v):
  '''
  Convierte un valor analítico a numero_
  - Si contiene '<', extra el LD y lo divide entre 2 (convención USEPA)
  - Reemplaza coma decimal por punto
  - Si no puede convertirse, devuelve NaN
  '''

  v = str(v).strip()
  v = v.replace(',','.')     #linea para cambiar la coma decimal por el punto
  if v.startswith('<'):
    try:
      lod = float(v.replace('<', '').strip())   #linea para extraer el número despues del '<'
      return lod/2
    except ValueError:
      return np.nan
  try:
    return float(v)
  except ValueError:
    return np.nan

#Tercero: aplicar la función a toda la columna Valor de cada dataset
df_sedimento['Valor_num'] = df_sedimento['Valor'].apply(limpiar_valor)

#Cuarto: imprimir resultado
print('Columna Valor_num creada')
print(f'Valores numéricos válidos (sedimento): {df_sedimento['Valor_num'].notna().sum()}')
print(f'Valores no convertibles (NaN) (sedimento): {df_sedimento['Valor_num'].isna().sum()}')

Columna Valor_num creada
Valores numéricos válidos (sedimento): 218
Valores no convertibles (NaN) (sedimento): 0


In [136]:
# FECHA viene como MM/DD/YY (int) -> lo convertimos a datetime real
df_suelo['Fecha'] = pd.to_datetime(df_suelo['Fecha'], format='%m/%d/%y', errors='coerce')

# Hora ya está en formato correcto de 0 a 23h

In [137]:
# Columnas finales que se van a usar para SEDIMENTO
# SI FALTAN COLUMNAS AÑADIR AQUÍ

final_cols_sedimento = ['Etapa','Componente ambiental','Nombre del punto','lat','lon','Fecha','Hora','Valor_num','Parámetro','Unidad de medida']

# Asegurarse de que todas las columnas finales existen antes de seleccionar
missing_cols = [col for col in final_cols_sedimento if col not in df_sedimento.columns]
if missing_cols:
    print(f"Advertencia: Las siguientes columnas faltan en df_sedimento y no pueden ser seleccionadas: {missing_cols}")

# Seleccionar solo las columnas deseadas que existen
df_sedimento = df_sedimento[[col for col in final_cols_sedimento if col in df_sedimento.columns]]

df_sedimento.head()

,Etapa,Componente ambiental,Nombre del punto,lat,lon,Fecha,Hora,Valor_num,Parámetro,Unidad de medida
0,Primer monitoreo,Sedimento,F05898_SD01,-4.453513,-81.284285,10/6/19,10:07:00,0.9500,Fracciones de Hidrocarburos F1 (C6-C10),mg/Kg
1,Primer monitoreo,Sedimento,F05898_SD01,-4.453513,-81.284285,10/6/19,10:07:00,3.4000,Fracciones de Hidrocarburos F2 (C10-C28),mg/Kg
2,Primer monitoreo,Sedimento,F05898_SD01,-4.453513,-81.284285,10/6/19,10:07:00,3.4000,Fracciones de Hidrocarburos F3 (C28-C40),mg/Kg
3,Primer monitoreo,Sedimento,F05898_SD01,-4.453513,-81.284285,10/6/19,10:07:00,0.0027,Acenafteno,mg/Kg
4,Primer monitoreo,Sedimento,F05898_SD01,-4.453513,-81.284285,10/6/19,10:07:00,0.0027,Acenaftileno,mg/Kg


### 3.3 Datos de AGUA

Orden:
1. Corregir ubicación --> Utilizar coordenadas
2. Corregir Fecha
3. Corregir Hora (formato 24h o mantener en AM y PM)
4. Crear columna de Valor_num
5. Extraer solo columna que se van a utilizar: Etapa, Componente ambiental,
Nombre del punto, Coordenadas, Fecha, Hora, Valor_num, Parámetro, Unidad de medida

In [138]:
# Nombre para CSV_PATH_AGUA

df_agua = pd.read_csv(CSV_PATH_AGUA)

df_agua.head()

,Número de informe,Nombre de la Evaluación,Etapa,Componente ambiental,Procedencia de la Muestra,Procedencia Especifica de la Muestra,Nombre del punto,Este,Norte,Altitud,Zona,Datum,Descripción de ubicación,Tipo de muestra,Tipo de análisis,Fecha,Hora,Valor,Parámetro,Unidad de medida
0,INFORME N° 00055-2019-OEFA/DEAM-SSIM,Identificación de Pasivo Ambiental del Subsect...,Segundo monitoreo,Agua,Agua natural superficial,Agua superficial de río,F01861-AG01,601347,9506175,76,18,WGS84,Punto ubicado en el río Tigre a 9 m de la oril...,Simple,Fisico Químicos,1/14/19,5:54:00 PM,<0.100,Aceites y grasas,mg/L
1,INFORME N° 00055-2019-OEFA/DEAM-SSIM,Identificación de Pasivo Ambiental del Subsect...,Segundo monitoreo,Agua,Agua natural superficial,Agua superficial de río,F01861-AG01,601347,9506175,76,18,WGS84,Punto ubicado en el río Tigre a 9 m de la oril...,Simple,Aniones por Cromatografía Iónica,1/14/19,5:54:00 PM,"0,233",Cloruros,mg/L
2,INFORME N° 00055-2019-OEFA/DEAM-SSIM,Identificación de Pasivo Ambiental del Subsect...,Segundo monitoreo,Agua,Agua natural superficial,Agua superficial de río,F01861-AG01,601347,9506175,76,18,WGS84,Punto ubicado en el río Tigre a 9 m de la oril...,Simple,Ensayo por cromatografía - PAHS,1/14/19,5:54:00 PM,<0.000013,Acenafteno,mg/L
3,INFORME N° 00055-2019-OEFA/DEAM-SSIM,Identificación de Pasivo Ambiental del Subsect...,Segundo monitoreo,Agua,Agua natural superficial,Agua superficial de río,F01861-AG01,601347,9506175,76,18,WGS84,Punto ubicado en el río Tigre a 9 m de la oril...,Simple,Ensayo por cromatografía - PAHS,1/14/19,5:54:00 PM,<0.000013,Acenaftileno,mg/L
4,INFORME N° 00055-2019-OEFA/DEAM-SSIM,Identificación de Pasivo Ambiental del Subsect...,Segundo monitoreo,Agua,Agua natural superficial,Agua superficial de río,F01861-AG01,601347,9506175,76,18,WGS84,Punto ubicado en el río Tigre a 9 m de la oril...,Simple,Ensayo por cromatografía - PAHS,1/14/19,5:54:00 PM,<0.000016,Antraceno,mg/L


In [139]:
## Limpieza de datos: Tratamiento de valores faltantes

#Con esta linea de códigos detectamos columnas con datos faltantes
df_agua_nulos = df_agua.isnull().sum()
print('Valores nulos por columna: Matriz Agua')
print(df_agua_nulos[df_agua_nulos > 0] if df_agua_nulos.any() else 'No hay valores nulos')

Valores nulos por columna: Matriz Agua
No hay valores nulos


In [140]:
##Exploración de datos: mapa de puntos de monitoreo por matriz
# Conversión de Unidades

from pyproj import Transformer

def utm_to_latlon_dynamic(row):
    # Determinar el sistema de referencias de coordenadas (CRS) UTM a partir de la columna 'Zona'
    # Suponiento el Hemisferio Sur para Perú (EPSG:327XX)
    utm_crs = f"epsg:327{int(row['Zona'])}" #Como las muestras pueden venir de diferentes UTM, esta función genera el EPSG dinámico
    # Transformer.from_crs expects source_crs, target_crs
    # always_xy=True ensures output is (longitude, latitude)
    transformer = Transformer.from_crs(utm_crs, "epsg:4326", always_xy=True)

    # 'Este' is Easting, 'Norte' is Northing
    lon_wgs84, lat_wgs84 = transformer.transform(row['Este'], row['Norte'])
    return pd.Series([lat_wgs84, lon_wgs84])

In [141]:
# Estas lineas son para aplicar la conversión dinámica al DataFrame.
# Despues hay que asegurarse que las columnas de 'Zona', 'Este', 'Norte' sean numéricas y esta linea gestiona posibles errores
for current_df in [df_agua]:
    cols_to_process = []
    if 'Este' in current_df.columns:
        cols_to_process.append('Este')
    if 'Norte' in current_df.columns:
        cols_to_process.append('Norte')
    if 'Zona' in current_df.columns: # Solo procesar 'Zona' si existe
        cols_to_process.append('Zona')

    for col in cols_to_process:
        current_df[col] = pd.to_numeric(current_df[col], errors='coerce')

    if cols_to_process: # Solo aplicar dropna si hay columnas a considerar
        current_df.dropna(subset=cols_to_process, inplace=True)

# Aplicar la conversión de UTM a Lat/Lon
# Solo aplicar si 'Zona', 'Este' y 'Norte' existen en el DataFrame
if all(col in df_agua.columns for col in ['Este', 'Norte', 'Zona']):
    df_agua[['lat', 'lon']] = df_agua.apply(utm_to_latlon_dynamic, axis=1) #Esta linea crea las coordenadas en el formato que son necesarias para Folium (agua)
else:
    print("Las columnas 'Este', 'Norte' o 'Zona' no están disponibles en df_agua, no se aplicó la conversión UTM a Lat/Lon.")

print("Coordenadas convertidas a WGS84 para agua (si las columnas estaban disponibles).")

# Mostrar el head de 'agua' con las nuevas coordenadas para verificación
if 'lat' in df_agua.columns:
    display(df_agua[['Nombre del punto', 'Este', 'Norte', 'Zona', 'lat', 'lon']].head())
else:
    print("No se pudieron generar las columnas 'lat' y 'lon' para df_agua.")

Coordenadas convertidas a WGS84 para agua (si las columnas estaban disponibles).


,Nombre del punto,Este,Norte,Zona,lat,lon
0,F01861-AG01,601347,9506175,18,-4.46713,-74.086502
1,F01861-AG01,601347,9506175,18,-4.46713,-74.086502
2,F01861-AG01,601347,9506175,18,-4.46713,-74.086502
3,F01861-AG01,601347,9506175,18,-4.46713,-74.086502
4,F01861-AG01,601347,9506175,18,-4.46713,-74.086502


In [142]:
# Crear columna Valor_num --> Correción de los valores numéricos

#Segundo: limpiamos los valores y lo reemplazamos por LD/2
import re
import numpy as np
import pandas as pd # Added for pd.read_csv and pd.Int64Dtype()

def limpiar_valor(v):
  '''
  Convierte un valor analítico a numero_
  - Si contiene '<', extra el LD y lo divide entre 2 (convención USEPA)
  - Reemplaza coma decimal por punto
  - Si no puede convertirse, devuelve NaN
  '''

  v = str(v).strip()
  v = v.replace(',','.')     #linea para cambiar la coma decimal por el punto
  if v.startswith('<'):
    try:
      lod = float(v.replace('<', '').strip())   #linea para extraer el número despues del '<'
      return lod/2
    except ValueError:
      return np.nan
  try:
    return float(v)
  except ValueError:
    return np.nan

#Tercero: aplicar la función a toda la columna Valor de cada dataset
df_agua['Valor_num'] = df_agua['Valor'].apply(limpiar_valor)

#Cuarto: imprimir resultado
print('Columna Valor_num creada')
print(f'Valores numéricos válidos (agua): {df_agua['Valor_num'].notna().sum()}')
print(f'Valores no convertibles (NaN) (agua): {df_agua['Valor_num'].isna().sum()}')

Columna Valor_num creada
Valores numéricos válidos (agua): 998
Valores no convertibles (NaN) (agua): 0


In [143]:
# FECHA viene como MM/DD/YY (int) -> lo convertimos a datetime real
df_agua['Fecha'] = pd.to_datetime(df_agua['Fecha'], format='%m/%d/%y', errors='coerce')

# Hora viene en formato AM/PM --> convertir a formato 0 a 23h
df_agua['Hora'] = pd.to_datetime(df_agua['Hora'], format='mixed', errors='coerce').dt.strftime('%H:%M')

In [144]:
# Columnas finales que se van a usar para AGUA
# SI FALTAN COLUMNAS AÑADIR AQUÍ

final_cols_agua = ['Etapa','Componente ambiental','Nombre del punto','lat','lon','Fecha','Hora','Valor_num','Parámetro','Unidad de medida']

# Asegurarse de que todas las columnas finales existen antes de seleccionar
missing_cols = [col for col in final_cols_agua if col not in df_agua.columns]
if missing_cols:
    print(f"Advertencia: Las siguientes columnas faltan en df_agua y no pueden ser seleccionadas: {missing_cols}")

# Seleccionar solo las columnas deseadas que existen
df_agua = df_agua[[col for col in final_cols_agua if col in df_agua.columns]]

df_agua.head()

,Etapa,Componente ambiental,Nombre del punto,lat,lon,Fecha,Hora,Valor_num,Parámetro,Unidad de medida
0,Segundo monitoreo,Agua,F01861-AG01,-4.46713,-74.086502,2019-01-14,17:54,0.050000,Aceites y grasas,mg/L
1,Segundo monitoreo,Agua,F01861-AG01,-4.46713,-74.086502,2019-01-14,17:54,0.233000,Cloruros,mg/L
2,Segundo monitoreo,Agua,F01861-AG01,-4.46713,-74.086502,2019-01-14,17:54,0.000006,Acenafteno,mg/L
3,Segundo monitoreo,Agua,F01861-AG01,-4.46713,-74.086502,2019-01-14,17:54,0.000006,Acenaftileno,mg/L
4,Segundo monitoreo,Agua,F01861-AG01,-4.46713,-74.086502,2019-01-14,17:54,0.000008,Antraceno,mg/L


## 4. Descarga de DATOS PROCESADOS (datos limpios)

Estos serán cargados en el Dashboard de visualización para que no se incluya la limpieza y procesamiento.

In [145]:
# Descargar df_suelo a un archivo CSV
df_suelo.to_csv('suelo_procesado.csv', index=False)
print('DataFrame df_suelo guardado en suelo_procesado.csv')
files.download('suelo_procesado.csv')

DataFrame df_suelo guardado en suelo_procesado.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [146]:
# Descargar df_sedimento a un archivo CSV
df_sedimento.to_csv('sedimento_procesado.csv', index=False)
print('DataFrame df_sedimento guardado en sedimento_procesado.csv')
files.download('sedimento_procesado.csv')

DataFrame df_sedimento guardado en sedimento_procesado.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [147]:
# Descargar df_agua a un archivo CSV
df_agua.to_csv('agua_procesado.csv', index=False)
print('DataFrame df_agua guardado en agua_procesado.csv')
files.download('agua_procesado.csv')

DataFrame df_agua guardado en agua_procesado.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>